# 7 · Silver, where the sources stop being separate

You now have five bronze tables. Each is a faithful copy of one source, shaped
the way that source happens to be shaped. **Nobody outside the team can use any
of them.**

Silver is one clean row per ride, with everything attached.

![](img/silver-1-shape.png)

| | |
|---|---|
| **reads** | all five bronze tables |
| **writes** | `teach.silver_rides` |

In [ ]:
import sys; sys.path.insert(0, '.')
from nb import show, sql, fetch, run, counts

import psycopg
from pipelines.lib.config import dsn, SCHEMA

counts()

---

## Step 0 · What each source can tell us about one ride

Pick a real ride and ask all five tables about it.

In [ ]:
trip_id = fetch(f"""
    SELECT t.trip_id FROM {SCHEMA}.bronze_trips t
    JOIN {SCHEMA}.bronze_settlements s ON s.trip_id = t.trip_id
    WHERE t.status = 'completed' LIMIT 1
""").trip_id[0]

print('looking at', trip_id, '\n')

with psycopg.connect(dsn()) as c:
    for table, q in [
        ('bronze_trips',       f'SELECT status, distance_km, duration_s, pickup_zone FROM {SCHEMA}.bronze_trips WHERE trip_id = %s'),
        ('bronze_events',      f'SELECT event, fare FROM {SCHEMA}.bronze_events WHERE trip_id = %s ORDER BY happened_at'),
        ('bronze_driver_app',  f'SELECT app_version, surge FROM {SCHEMA}.bronze_driver_app WHERE trip_id = %s'),
        ('bronze_settlements', f'SELECT status, gross, fee, net FROM {SCHEMA}.bronze_settlements WHERE trip_id = %s'),
    ]:
        rows = c.execute(q, (trip_id,)).fetchall()
        print(f'{table}')
        for r in rows or [('nothing',)]:
            print('   ', r)
        print()

**Four tables, four shapes, one ride.** One of them has five rows for it. One
has none. That is the problem silver exists to solve.

---

## Step 1 · The join, and the most expensive word in this job

![](img/silver-2-join.png)

Every join below is a `LEFT JOIN`, and every one is deliberate.

In [ ]:
BUILD = f"""
SELECT
    t.trip_id,
    t.trip_date,
    t.status,
    t.driver_id,
    t.pickup_zone,
    z.zone_name                     AS pickup_zone_name,
    z.borough                       AS pickup_borough,
    t.distance_km,
    round(t.duration_s / 60.0, 1)   AS duration_min,
    e.fare,
    a.surge,
    s.net                           AS settled_net,
    (s.settlement_id IS NOT NULL)   AS is_settled
FROM {SCHEMA}.bronze_trips t
LEFT JOIN {SCHEMA}.bronze_events e
       ON e.trip_id = t.trip_id
      AND e.event = 'completed'     -- in the ON, never the WHERE. See below.
LEFT JOIN {SCHEMA}.bronze_driver_app a
       ON a.trip_id = t.trip_id
LEFT JOIN {SCHEMA}.bronze_settlements s
       ON s.trip_id = t.trip_id
      AND s.status = 'settled'
LEFT JOIN {SCHEMA}.bronze_zones z
       ON z.zone_id = t.pickup_zone
"""

print(BUILD)

### Now count what each choice actually costs

This is the cell to put on the projector.

In [ ]:
with psycopg.connect(dsn()) as c:
    left_n = c.execute(f'SELECT count(*) FROM ({BUILD}) x').fetchone()[0]

    inner_n = c.execute(f"""
        SELECT count(*) FROM {SCHEMA}.bronze_trips t
        JOIN {SCHEMA}.bronze_events e
          ON e.trip_id = t.trip_id AND e.event = 'completed'
        JOIN {SCHEMA}.bronze_driver_app a  ON a.trip_id = t.trip_id
        JOIN {SCHEMA}.bronze_settlements s ON s.trip_id = t.trip_id AND s.status = 'settled'
        JOIN {SCHEMA}.bronze_zones z       ON z.zone_id = t.pickup_zone
    """).fetchone()[0]

print(f'LEFT  JOIN : {left_n:>8,} rides')
print(f'INNER JOIN : {inner_n:>8,} rides')
print(f'\ndifference : {left_n - inner_n:>8,} rides, {(left_n - inner_n) / left_n:.1%}, gone')

### Where exactly did they go?

Four separate reasons, and every one of them is a real ride.

In [ ]:
qs = {
 'no completed event (cancelled rides)':
   f"""SELECT count(*) FROM {SCHEMA}.bronze_trips t WHERE NOT EXISTS (
        SELECT 1 FROM {SCHEMA}.bronze_events e
        WHERE e.trip_id = t.trip_id AND e.event = 'completed')""",
 'no driver app record':
   f"""SELECT count(*) FROM {SCHEMA}.bronze_trips t WHERE NOT EXISTS (
        SELECT 1 FROM {SCHEMA}.bronze_driver_app a WHERE a.trip_id = t.trip_id)""",
 'not settled yet (settlement is T+2)':
   f"""SELECT count(*) FROM {SCHEMA}.bronze_trips t WHERE NOT EXISTS (
        SELECT 1 FROM {SCHEMA}.bronze_settlements s
        WHERE s.trip_id = t.trip_id AND s.status = 'settled')""",
 'never resolved to a zone':
   f"""SELECT count(*) FROM {SCHEMA}.bronze_trips t WHERE NOT EXISTS (
        SELECT 1 FROM {SCHEMA}.bronze_zones z WHERE z.zone_id = t.pickup_zone)""",
}

with psycopg.connect(dsn()) as c:
    for label, q in qs.items():
        print(f'  {c.execute(q).fetchone()[0]:>7,}   {label}')

### Read that list again

A cancelled ride **is a ride**. A ride the app did not report **happened**.
Money that has not settled yet **will**. And a ride with no zone id **still
carried somebody somewhere**.

An `INNER JOIN` deletes all four, and:

> **No error. No failed run. No log line. The number is just smaller.**

Somebody changes a join to make a query faster, the ride count drops four
percent, and it takes three weeks to notice and a day to find.

**A wrong answer that runs successfully is worse than a crash, because a crash
tells you.**

---

## Step 2 · ON, or WHERE. Not a style choice.

![](img/silver-3-where.png)

Same condition, one word moved, and the LEFT JOIN quietly becomes an INNER one.
Prove it.

In [ ]:
with psycopg.connect(dsn()) as c:
    on_clause = c.execute(f"""
        SELECT count(*) FROM {SCHEMA}.bronze_trips t
        LEFT JOIN {SCHEMA}.bronze_events e
               ON e.trip_id = t.trip_id AND e.event = 'completed'
    """).fetchone()[0]

    where_clause = c.execute(f"""
        SELECT count(*) FROM {SCHEMA}.bronze_trips t
        LEFT JOIN {SCHEMA}.bronze_events e
               ON e.trip_id = t.trip_id
        WHERE e.event = 'completed'
    """).fetchone()[0]

print(f"condition in ON    : {on_clause:>8,}")
print(f"condition in WHERE : {where_clause:>8,}")
print(f"\n{on_clause - where_clause:,} rides lost to moving one word")

**The WHERE runs after the join.** The LEFT JOIN carefully kept the rides with
no completed event and gave them `NULL`, and then the WHERE threw every one of
those NULL rows away.

Same rows lost as an INNER JOIN, with a `LEFT JOIN` sitting in the query looking
reassuring.

---

## Step 3 · The table silver writes into

In [ ]:
DDL = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.silver_rides (
    trip_id      TEXT PRIMARY KEY,
    trip_date    DATE NOT NULL,
    status       TEXT,
    driver_id    TEXT,
    pickup_zone  INT,
    pickup_zone_name TEXT,        -- from the zones dimension
    pickup_borough   TEXT,        -- an id nobody can read is not an answer
    distance_km  NUMERIC(8,2),
    duration_min NUMERIC(8,1),    -- converted from seconds
    fare         NUMERIC(10,2),   -- from the event stream
    surge        NUMERIC(6,2),    -- from the driver app
    settled_net  NUMERIC(12,2),   -- from the payment processor
    is_settled   BOOLEAN          -- a plain yes/no, so nobody downstream guesses
);
CREATE INDEX IF NOT EXISTS ix_silver_rides_date ON {SCHEMA}.silver_rides (trip_date);
"""

with psycopg.connect(dsn(), autocommit=True) as c:
    c.execute(DDL)

print('table ready')

### Three things silver did that bronze was forbidden to do

| | |
|---|---|
| `duration_s` became `duration_min` | **a unit conversion.** Bronze was not allowed one |
| `fare`, `surge`, `settled_net` | **columns that existed in no single source.** The whole point |
| no `rider_id` | **dropped.** Nothing downstream uses it, and carrying a personal identifier you do not need is a liability, not an asset |

---

## Step 4 · Build it, in one transaction

In [ ]:
with psycopg.connect(dsn(), autocommit=False) as c, c.cursor() as cur:
    rows_in = cur.execute(f'SELECT count(*) FROM {SCHEMA}.bronze_trips').fetchone()[0]

    # Both statements inside one transaction. A crash between them rolls back
    # the delete too, so a reader never sees an empty table.
    cur.execute(f'DELETE FROM {SCHEMA}.silver_rides')
    cur.execute(f'INSERT INTO {SCHEMA}.silver_rides {BUILD}')
    rows_out = cur.rowcount
    c.commit()

print(f'read {rows_in:,} rides, wrote {rows_out:,}')

### Why a full rebuild is defensible here, and would not be at scale

Bronze rebuilt **a window** because the source table is enormous. Here we delete
everything and rebuild, which would be indefensible at real scale.

It is fine here because `silver_rides` holds eighty thousand rows and the
rebuild takes under a second. Being clever about incremental windows would cost
more in explanation than it saves in runtime.

When this table reaches tens of millions it grows a window, exactly like bronze
did. The honest rule is not *"always full rebuild"*, it is:

> **Match the technique to the size, and say out loud which one you chose.**

---

## Step 5 · Look at what you built

In [ ]:
sql(f"""
    SELECT trip_id, trip_date, status, pickup_zone_name, pickup_borough,
           distance_km, duration_min, fare, surge, settled_net, is_settled
    FROM {SCHEMA}.silver_rides
    WHERE fare IS NOT NULL
    ORDER BY trip_date DESC
    LIMIT 8
""", 'silver_rides, one row per ride')

**A person who has never heard of Kafka can read that table.** That is the test
silver has to pass.

## How complete is it?

Silver keeps the ride even when a source had nothing. So say plainly how often
each column is actually filled.

In [ ]:
sql(f"""
    SELECT count(*)                                                   AS rides,
           round(100.0 * count(fare)        / count(*), 1) AS pct_with_fare,
           round(100.0 * count(surge)       / count(*), 1) AS pct_with_surge,
           round(100.0 * count(settled_net) / count(*), 1) AS pct_settled,
           round(100.0 * count(pickup_zone_name) / count(*), 1) AS pct_with_zone
    FROM {SCHEMA}.silver_rides
""", 'how filled in is each column')

**Those gaps are the truth, not a bug.** Every one of them has a reason you
counted a moment ago. A table that reported 100% everywhere would be lying.

---

## And the packaged pipeline

In [ ]:
run('-m', 'pipelines.p7_silver_rides')

In [ ]:
sql(f"""
    SELECT pipeline, status, rows_in, rows_out,
           round(extract(epoch from (ended_at - started_at))::numeric, 2) AS secs, message
    FROM {SCHEMA}.runs WHERE pipeline = 'p7_silver_rides'
    ORDER BY started_at DESC LIMIT 3
""", 'the run log')

---

## What you learned

- Silver is **one clean row per thing**, and the first table a stranger can read
- Silver **may** convert units, add derived columns, and drop what nothing needs
- **`LEFT` versus `INNER` is silent data loss.** Count it before you choose
- A cancelled ride, an unreported ride, an unsettled ride and a zoneless ride
  are all still rides
- Put the condition in the **`ON`**, never the `WHERE`, or your LEFT JOIN is a
  lie
- Full rebuild or window is **a size decision**, and you should say which you made
- **Report your gaps.** 100% everywhere means somebody is not looking